# Baseline Model Training

## baseline_train — Process Flow Outline

- **0. Goal / Outputs**
  - Reproducible baseline (fixed split/CV, metrics, seeds)
  - Comparable CV results
  - Final test report + next-stage improvement list

- **1. Data Preprocessing (sklearn)**
  - Define: target, feature columns, numeric vs categorical
  - Build `ColumnTransformer` + `Pipeline`
  - Fit transforms on **train only** (avoid leakage)
  - Output: processed features + saved preprocessing pipeline

- **2. Cross Validation (PyTorch Logistic Baseline)**
  - CV: `StratifiedKFold` (classification)
  - Model: Logistic Regression equivalent (linear layer + sigmoid/softmax)
  - Training sweep: `learning_rate`, `weight_decay`
  - Track: train/val loss curves + primary metric
  - Output: locked evaluation protocol (metric, threshold, CV config)

- **3. Train Baseline Model**
  - Use the same model as Step 2
  - Train on full train set with selected hyperparameters
  - Output: baseline performance (CV mean±std if applicable) + learning curves

- **4. Hyperparameter Optimization**
  - Keep the same CV framework
  - Search method: random / bayes / Optuna
  - Search space (examples): lr, wd, batch size, epochs, class weights
  - Output: best config + validation score summary

- **5. Final Evaluation (Test) + Next Steps**
  - Run test evaluation once (no tuning on test)
  - Report: primary metric + confusion matrix (and calibration if needed)
  - Error analysis: weak classes/slices, imbalance, redundancy
  - Next-stage plan: stronger models, feature engineering, thresholding, data work


## Package Import & Path Setting

In [1]:
from pathlib import Path
import pandas as pd
from pandas import DataFrame as df
import numpy as np
import matplotlib.pyplot as plt

# data split
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# preprocessing & structure
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# metrics
from sklearn.metrics import accuracy_score

# pytorch
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# other imports
from tqdm.auto import tqdm


In [2]:
PROJECT_ROOT = Path.cwd().parents[0]
DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_data_path = DATA_DIR / 'train.csv'
test_data_path = DATA_DIR / 'test.csv'
test_label_path = DATA_DIR / 'gender_submission.csv'

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data Loading

In [3]:
train_raw_df = pd.read_csv(train_data_path)
test_raw_df = pd.read_csv(test_data_path)
test_label_df = pd.read_csv(test_label_path)

X = train_raw_df.drop('Survived', axis=1)
y = train_raw_df['Survived'] 

## Data Preprocessing

* Columns to drop: `Name`, `Cabin`, `Ticket`
* Numeral columns: `Age`, `Fare`, `SibSp`, `Parch`
* Numeral columns to be scaled: `Age`, `Fare`
* Categorical columns: `Sex`, `Pclass`, `Embarked`

In [4]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
cat_cols = ['Sex', 'Pclass', 'Embarked']

def create_new_preprocess():
    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore')),
            ]), cat_cols),
        ],
        remainder='drop'
    )
    return preprocess

* A data preprocessing pipeline is constructed using `ColumnTransformer`.
* The weak impact features `Embarked` and `Parch` are included, and they will be the subjects to feature ablation.

## Cross Validation

### Baseline Model

In [5]:
# baseline model
class LogisticClass(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.linear(x).squeeze(1)

 
# training class
class Trainer():
    def __init__(self, model, optimizer, loss_fn, metrics, device=None):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metrics = metrics
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.hist = {"loss": [], "val_loss": []}
        for name in self.metrics.keys():
            self.hist[name] = []        

    def train(self, train_data, val_data, batch_size, epochs):
        X_tr_np, y_tr_np = train_data
        X_val_np, y_val_np = val_data
        
        X_tr = torch.tensor(X_tr_np, dtype=torch.float32)
        y_tr = torch.tensor(y_tr_np, dtype=torch.float32)
        X_val = torch.tensor(X_val_np, dtype=torch.float32)
        y_val = torch.tensor(y_val_np, dtype=torch.float32)

        if y_tr.ndim == 1:
            y_tr = y_tr.view(-1, 1)
        if y_val.ndim == 1:
            y_val = y_val.view(-1, 1)

        train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)             

        # for epoch in range(epochs):
        for epoch in tqdm(range(epochs), desc="Training", leave=False):
            self.model.train()
            total_loss = 0
            total_n = 0
            for x, y in train_loader:
                x = x.to(self.device)
                y = y.to(self.device)
                self.optimizer.zero_grad()
                logits = self.model(x).squeeze()
                loss = self.loss_fn(logits, y.squeeze())
                actual_bs = x.size(0)
                total_loss += loss.item() * actual_bs
                total_n += actual_bs
                loss.backward()
                self.optimizer.step()
            self.hist['loss'].append(total_loss / max(total_n, 1))            
            # validation
            self.model.eval()
            val_total_loss = 0
            val_total_n = 0            
            probs_list = []
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(self.device)
                    y = y.to(self.device)
                    logits = self.model(x).squeeze()
                    loss = self.loss_fn(logits, y.squeeze())
                    actual_bs = x.size(0)
                    val_total_loss += loss.item() * actual_bs
                    val_total_n += actual_bs
                    p = torch.sigmoid(logits).detach().cpu().numpy()
                    probs_list.append(p)
            
            self.hist["val_loss"].append(val_total_loss / max(val_total_n, 1))
            probs = np.concatenate(probs_list, axis=0)
            probs_1d = probs.reshape(-1)

            for name, spec in self.metrics.items():
                if spec["input_type"] == "prob":
                    self.hist[name].append(spec["func"](y_val, probs_1d))
                elif spec["input_type"] == "label":
                    pred = (probs_1d >= 0.5).astype(int)
                    self.hist[name].append(spec["func"](y_val, pred))
                else:
                    raise ValueError(f"Unknown input_type for metric '{name}': {spec['input_type']}")
        return self.hist
    

In [8]:
## skf

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=37)
batch_size = 64
epochs = 5
lr = 1e-2
weight_decay=0.0
loss_fn = nn.BCEWithLogitsLoss()

metrics = {'acc':{'input_type':'label', 'func':accuracy_score}}
scores = []
dummy_scores = []
# for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
for fold, (tr_idx, val_idx) in enumerate(tqdm(skf.split(X, y), total=skf.get_n_splits(), desc="CV folds"), start=1):
    X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    preprocess = create_new_preprocess()
    X_tr = preprocess.fit_transform(X_tr_raw)
    X_val = preprocess.transform(X_val_raw)
    train_data = X_tr, y_tr.to_numpy()
    val_data = X_val, y_val.to_numpy()
    model = LogisticClass(12).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    trainer = Trainer(model, optimizer, loss_fn, metrics, device=DEVICE)
    hist = trainer.train(train_data, val_data, batch_size, epochs)
    with torch.no_grad():
        X_val = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        logits = trainer.model(X_val)
        p = torch.sigmoid(logits).cpu().numpy()
    pred = (p >= 0.5).astype(int).squeeze()
    score = accuracy_score(y_val, pred)
    scores.append(score)
    maj_label = int(np.bincount(np.asarray(y_val, dtype=int)).argmax())
    dummy_pred = np.full(len(y_val), maj_label, dtype=int)
    dummy_score = accuracy_score(y_val, dummy_pred)
    dummy_scores.append(dummy_score)
    print(f"Fold {fold} baseline score: {score:.4f}, dummy score: {dummy_score:.4f}")

print(f"Baseline Model Mean: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
print(f"Dummy Model Mean: {np.mean(dummy_scores):.4f} ± {np.std(dummy_scores):.4f}")
print(f'Baseline Model Lift: {(np.mean(scores) - np.mean(dummy_scores)):.4f}')

CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1 baseline score: 0.7318, dummy score: 0.6145


Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 2 baseline score: 0.8090, dummy score: 0.6180


...

Fold 3 baseline score: 0.8202, dummy score: 0.6180


Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 4 baseline score: 0.7640, dummy score: 0.6180


Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 5 baseline score: 0.7303, dummy score: 0.6124
Baseline Model Mean: 0.7711 ± 0.0377
Dummy Model Mean: 0.6162 ± 0.0023
Baseline Model Lift: 0.1549
